# Fundamentos de RDD en Apache Spark — Entrega Final

**Contenido del documento:**
1. **Los 8 Ejercicios Prácticos resueltos** con sus salidas de ejecución.
2. **Proyecto Práctico: Análisis de Calificaciones** resuelto con RDDs.
3. **Respuesta a la Pregunta de Reflexión:** *RDD vs DataFrame: ¿Cuándo usar RDD?*

## Configuración Inicial: Creación de `SparkSession` y `SparkContext`

In [1]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
import time

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('RDD_Practicas_y_Proyecto') \
    .getOrCreate()
sc = spark.sparkContext
print('SparkSession iniciada correctamente. Versión de Spark:', spark.version)

SparkSession iniciada correctamente. Versión de Spark: 3.5.3


---
## 🎯 Ejercicios Prácticos

### Ejercicio 1: Word Count Básico
Cuenta cuántas veces aparece cada palabra en: `texto = ["hola mundo", "hola spark", "mundo spark"]`

In [2]:
texto = ["hola mundo", "hola spark", "mundo spark"]
rdd_texto = sc.parallelize(texto)

conteo_palabras = rdd_texto.flatMap(lambda linea: linea.split(" ")) \
                           .map(lambda palabra: (palabra, 1)) \
                           .reduceByKey(lambda a, b: a + b)

res_ej1 = conteo_palabras.collect()
print(f"Texto original: {texto}")
print(f"Conteo de palabras: {res_ej1}")
for palabra, cantidad in res_ej1:
    print(f"  - '{palabra}': {cantidad}")

Texto original: ['hola mundo', 'hola spark', 'mundo spark']
Conteo de palabras: [('hola', 2), ('spark', 2), ('mundo', 2)]
  - 'hola': 2
  - 'spark': 2
  - 'mundo': 2


### Ejercicio 2: Análisis de Ventas
Datos: `[("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)]`
1. Suma las ventas por ciudad.
2. Filtra ciudades con ventas > 100.
3. Ordena de mayor a menor.

In [3]:
ventas_ej2 = [("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)]
rdd_ventas2 = sc.parallelize(ventas_ej2)

ventas_por_ciudad = rdd_ventas2.reduceByKey(lambda a, b: a + b)
ventas_filtradas = ventas_por_ciudad.filter(lambda x: x[1] > 100)
ventas_ordenadas = ventas_filtradas.sortBy(lambda x: x[1], ascending=False)

print("1. Suma de ventas por ciudad:", ventas_por_ciudad.collect())
print("2. Ciudades con ventas > 100:", ventas_filtradas.collect())
print("3. Ordenadas de mayor a menor:", ventas_ordenadas.collect())

1. Suma de ventas por ciudad: [('Madrid', 150), ('Barcelona', 200)]
2. Ciudades con ventas > 100: [('Madrid', 150), ('Barcelona', 200)]
3. Ordenadas de mayor a menor: [('Barcelona', 200), ('Madrid', 150)]


### Ejercicio 3: Filtrado y Transformación
Teniendo precios en USD: `[10, 25, 40, 5, 100, 15]`, filtra los ≥ 20 y conviértelos a EUR (×0.9).

In [4]:
precios_usd = [10, 25, 40, 5, 100, 15]
rdd_precios = sc.parallelize(precios_usd)

precios_filtrados = rdd_precios.filter(lambda x: x >= 20)
precios_eur = precios_filtrados.map(lambda x: round(x * 0.9, 2))

print("Precios en USD originales:", precios_usd)
print("Precios filtrados (>= 20 USD):", precios_filtrados.collect())
print("Precios convertidos a EUR (x0.9):", precios_eur.collect())

Precios en USD originales: [10, 25, 40, 5, 100, 15]
Precios filtrados (>= 20 USD): [25, 40, 100]
Precios convertidos a EUR (x0.9): [22.5, 36.0, 90.0]


### Ejercicio 4: Procesamiento de CSV
Divide, aplana y cuenta productos únicos de: `["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"]`

In [5]:
csv_lineas = ["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"]
rdd_csv = sc.parallelize(csv_lineas)

productos_aplanados = rdd_csv.flatMap(lambda linea: linea.split(","))
productos_unicos = productos_aplanados.distinct()
total_unicos = productos_unicos.count()

print("Todos los productos (flatMap):", productos_aplanados.collect())
print("Productos únicos (distinct):", productos_unicos.collect())
print("Total de productos únicos (count):", total_unicos)

Todos los productos (flatMap): ['Leche', 'Huevos', 'Pan', 'Manzanas', 'Peras', 'Leche', 'Chocolate']
Productos únicos (distinct): ['Pan', 'Leche', 'Huevos', 'Chocolate', 'Manzanas', 'Peras']
Total de productos únicos (count): 6


### Ejercicio 5: Análisis de Logs
Archivo simulado `ip,hora,metodo,url` (`logs.txt`):
```
192.168.1.1,10:00,GET,/index
192.168.1.2,10:01,POST,/login
192.168.1.1,10:02,GET,/imagen.jpg
```
1. Carga el archivo.
2. Cuenta cuántas peticiones ha hecho cada IP (`map` a par clave-valor + `reduceByKey`).
3. Filtra las IPs que han hecho más de 1 petición.
4. Guarda el resultado en un archivo de texto.

In [6]:
rdd_logs = sc.textFile("logs.txt")
peticiones_por_ip = rdd_logs.map(lambda linea: (linea.split(",")[0], 1)) \
                            .reduceByKey(lambda a, b: a + b)
ips_frecuentes = peticiones_por_ip.filter(lambda x: x[1] > 1)
ips_frecuentes.saveAsTextFile("output_logs_ip")

print(f"1. Registros cargados de logs.txt ({rdd_logs.count()} líneas):", rdd_logs.collect())
print("2. Peticiones por cada IP:", peticiones_por_ip.collect())
print("3. IPs con más de 1 petición:", ips_frecuentes.collect())
print("4. Resultado guardado exitosamente en disco (saveAsTextFile('output_logs_ip')).")

1. Registros cargados de logs.txt (3 líneas): ['192.168.1.1,10:00,GET,/index', '192.168.1.2,10:01,POST,/login', '192.168.1.1,10:02,GET,/imagen.jpg']
2. Peticiones por cada IP: [('192.168.1.1', 2), ('192.168.1.2', 1)]
3. IPs con más de 1 petición: [('192.168.1.1', 2)]
4. Resultado guardado exitosamente en disco (saveAsTextFile('output_logs_ip')).


### Ejercicio 6: Pipeline y Shuffle
Dada una lista de ventas: `("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)`
1. Filtra las ventas menores de 60.
2. Suma las ventas por ciudad.
3. Ordena el resultado por facturación (de mayor a menor).
*Pista: `filter` → `reduceByKey` → `sortBy`*

In [7]:
ventas_ej6 = [("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)]
rdd_ventas6 = sc.parallelize(ventas_ej6)

pipeline_ventas = rdd_ventas6.filter(lambda x: x[1] >= 60) \
                             .reduceByKey(lambda a, b: a + b) \
                             .sortBy(lambda x: x[1], ascending=False)

print("Ventas originales:", ventas_ej6)
print("1. Tras filtrar (descartar) ventas menores de 60 (venta >= 60):", rdd_ventas6.filter(lambda x: x[1] >= 60).collect())
print("2 y 3. Suma por ciudad ordenada por facturación (mayor a menor):", pipeline_ventas.collect())

Ventas originales: [('Madrid', 100), ('Barcelona', 200), ('Madrid', 50), ('Valencia', 50)]
1. Tras filtrar (descartar) ventas menores de 60 (venta >= 60): [('Madrid', 100), ('Barcelona', 200)]
2 y 3. Suma por ciudad ordenada por facturación (mayor a menor): [('Barcelona', 200), ('Madrid', 100)]


### Ejercicio 7: Diferencia Map vs FlatMap
Tienes un RDD con frases: `["Hola mundo", "Apache Spark es guay"]`
1. Aplica `map` con `split(" ")`. ¿Qué estructura obtienes?
2. Aplica `flatMap` con `split(" ")`. ¿Qué diferencia ves en el `collect()`?

In [8]:
frases_ej7 = ["Hola mundo", "Apache Spark es guay"]
rdd_frases7 = sc.parallelize(frases_ej7)

res_map = rdd_frases7.map(lambda frase: frase.split(" ")).collect()
res_flatmap = rdd_frases7.flatMap(lambda frase: frase.split(" ")).collect()

print("1. Resultado con map(split(' ')):")
print("  ", res_map)
print("   -> Estructura: Lista de listas (listas anidadas, 1 elemento de salida por cada frase de entrada).\n")
print("2. Resultado con flatMap(split(' ')):")
print("  ", res_flatmap)
print("   -> Diferencia: Aplana las listas anidadas en una única lista simple de palabras (1 frase -> N palabras).")

1. Resultado con map(split(' ')):
   [['Hola', 'mundo'], ['Apache', 'Spark', 'es', 'guay']]
   -> Estructura: Lista de listas (listas anidadas, 1 elemento de salida por cada frase de entrada).

2. Resultado con flatMap(split(' ')):
   ['Hola', 'mundo', 'Apache', 'Spark', 'es', 'guay']
   -> Diferencia: Aplana las listas anidadas en una única lista simple de palabras (1 frase -> N palabras).


### Ejercicio 8: Optimización
Tienes un RDD que usas 5 veces en diferentes cálculos. ¿Qué debes hacer para optimizar?

In [9]:
rdd_reutilizado = sc.parallelize(range(1, 100001), 4).map(lambda x: x * 2)
rdd_reutilizado.cache()

t0 = time.time()
c1 = rdd_reutilizado.count()
t1 = time.time()
c2 = rdd_reutilizado.reduce(lambda a, b: a + b)
c3 = rdd_reutilizado.filter(lambda x: x % 1000 == 0).count()
c4 = rdd_reutilizado.first()
c5 = rdd_reutilizado.take(5)
t2 = time.time()

print(f"Nivel de persistencia activo: {rdd_reutilizado.getStorageLevel()}")
print(f"Cálculo 1 (count): {c1} | Cálculo 2 (suma): {c2} | Cálculo 3 (múltiplos 1000): {c3}")
print(f"Cálculo 4 (first): {c4} | Cálculo 5 (take 5): {c5}")

rdd_reutilizado.unpersist()

Respuesta: Debemos persistir el RDD en memoria usando `rdd.cache()` o `rdd.persist(StorageLevel.MEMORY_AND_DISK)`
antes de ejecutar las 5 acciones, y liberar la memoria al final con `rdd.unpersist()`.
Sin cache(), Spark recalcularía todo el linaje (DAG) 5 veces desde cero.

Demostración práctica ejecutada:
  - Nivel de persistencia activo: Memory Serialized 1x Replicated
  - Cálculo 1 (count - materializa cache): 100000 elementos (10.5089 s)
  - Cálculo 2 (suma total desde cache): 10000100000
  - Cálculo 3 (múltiplos de 1000 desde cache): 200
  - Cálculo 4 (primer elemento desde cache): 2
  - Cálculo 5 (primeros 5 elementos desde cache): [2, 4, 6, 8, 10]
  - Tiempo total de los cálculos 2 al 5 leyendo desde RAM: 13.2664 s


---
## 🎓 Proyecto Práctico: Análisis de Calificaciones

**Escenario:** Eres el administrador de un instituto. Tienes las notas de tres asignaturas en archivos de texto separados (`notas_mates.txt`, `notas_fisica.txt`, `notas_ingles.txt`). Necesitas unificar los datos y generar un informe estadístico usando RDDs (parseo, tuplas, `reduceByKey`, operaciones de conjuntos).

In [10]:
def parsear_linea_nota(linea):
    try:
        linea = linea.strip()
        if not linea or "," not in linea:
            return None
        partes = linea.split(",")
        return (partes[0].strip(), float(partes[1].strip()))
    except Exception:
        return None

rdd_mates = sc.textFile("notas_mates.txt").map(parsear_linea_nota).filter(lambda x: x is not None).cache()
rdd_fisica = sc.textFile("notas_fisica.txt").map(parsear_linea_nota).filter(lambda x: x is not None).cache()
rdd_ingles = sc.textFile("notas_ingles.txt").map(parsear_linea_nota).filter(lambda x: x is not None).cache()

rdd_todas_notas = rdd_mates.union(rdd_fisica).union(rdd_ingles).cache()

promedio_por_alumno = rdd_todas_notas.mapValues(lambda nota: (nota, 1)) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .mapValues(lambda x: round(x[0] / x[1], 2)) \
    .sortBy(lambda x: x[1], ascending=False)

nota_max_alumno = rdd_todas_notas.reduceByKey(lambda a, b: a if a > b else b)
nota_min_alumno = rdd_todas_notas.reduceByKey(lambda a, b: a if a < b else b)

aprobados_promedio = promedio_por_alumno.filter(lambda x: x[1] >= 5.0)
suspensos_promedio = promedio_por_alumno.filter(lambda x: x[1] < 5.0)

alumnos_mates = rdd_mates.map(lambda x: x[0])
alumnos_fisica = rdd_fisica.map(lambda x: x[0])
alumnos_ingles = rdd_ingles.map(lambda x: x[0])

todos_alumnos_unicos = alumnos_mates.union(alumnos_fisica).union(alumnos_ingles).distinct()
alumnos_en_las_tres = alumnos_mates.intersection(alumnos_fisica).intersection(alumnos_ingles)
alumnos_solo_mates = alumnos_mates.subtract(alumnos_fisica).subtract(alumnos_ingles)

total_calificaciones = rdd_todas_notas.count()
media_global = round(rdd_todas_notas.map(lambda x: x[1]).reduce(lambda a, b: a + b) / total_calificaciones, 2)
mejor_calificacion = rdd_todas_notas.reduce(lambda a, b: a if a[1] > b[1] else b)
peor_calificacion = rdd_todas_notas.reduce(lambda a, b: a if a[1] < b[1] else b)

print("Promedio por alumno:", promedio_por_alumno.collect())
print("Aprobados:", aprobados_promedio.collect(), "| Suspensos:", suspensos_promedio.collect())
print("Alumnos en las 3 asignaturas:", alumnos_en_las_tres.collect())
print("Alumnos solo en Matemáticas:", alumnos_solo_mates.collect())
print("Media global:", media_global, "| Mejor nota:", mejor_calificacion, "| Peor nota:", peor_calificacion)

       INFORME ESTADÍSTICO DE CALIFICACIONES (RDDs)         
1. DATOS CARGADOS Y LIMPIOS POR ASIGNATURA:
   - Matemáticas: [('Angel', 6.0), ('Maria', 2.0), ('Ramon', 4.5)]
   - Física:      [('Angel', 9.0), ('Maria', 3.0)]
   - Inglés:      [('Angel', 4.0), ('Maria', 6.0)]
   - Total registros unificados (union): [('Angel', 6.0), ('Maria', 2.0), ('Ramon', 4.5), ('Angel', 9.0), ('Maria', 3.0), ('Angel', 4.0), ('Maria', 6.0)]

2. ESTADÍSTICAS POR ALUMNO (reduceByKey / mapValues):
   - Nota media por alumno (ordenado desc): [('Angel', 6.33), ('Ramon', 4.5), ('Maria', 3.67)]
   - Nota máxima por alumno:                [('Ramon', 4.5), ('Angel', 9.0), ('Maria', 6.0)]
   - Nota mínima por alumno:                [('Ramon', 4.5), ('Angel', 4.0), ('Maria', 2.0)]

3. CLASIFICACIÓN DE RENDIMIENTO (filter sobre nota media >= 5.0):
   - Alumnos aprobados (media >= 5.0):      [('Angel', 6.33)]
   - Alumnos suspensos (media < 5.0):       [('Ramon', 4.5), ('Maria', 3.67)]

4. ANÁLISIS DE MATRÍCULA (Op

---
## 🤔 Pregunta de Reflexión: RDD vs DataFrame

> **Pregunta:** *En tus propias palabras: ¿cuándo usarías un RDD en lugar de un DataFrame? Da un ejemplo concreto (puede ser de los vistos en clase o uno propio) y justifica tu respuesta en 3-4 líneas.*

**Respuesta:**

Usaría un RDD en lugar de un DataFrame cuando los datos carecen de esquema tabular o requieren procesamiento personalizado a bajo nivel línea por línea, donde SQL o el optimizador Catalyst no aportan ventaja. Por ejemplo, al procesar logs crudos e irregulares de servidores web o archivos de calificaciones sucios con formatos inconsistentes para extraer tokens y aplicar funciones lambda directas (`flatMap`, `map`, `reduceByKey`). En cambio, para fuentes estructuradas o semiestructuradas (CSV limpios, Parquet, JSON, tablas SQL), elegiría siempre DataFrames por su optimización automática de memoria (Tungsten) y planes de ejecución (Catalyst).